# 1001 22일차

## 1. 임베딩 모델

문장을 받아 의미를 담은 벡터 하나로 바꿔 주는, 이미 학습된 모델

문장끼리 비교하려면 숫자로 바꿔야 하는데, 직접 학습하지 않고 학습된 모델에 맡김

1. `OpenAIEmbeddings` : OpenAI 임베딩 모델을 LangChain으로 부르는 클래스. `base_url`로 호환 서버(monogpt)에도 연결
2. `embed_query` : 문장 하나를 벡터 하나로 바꿈

| 모델 | 기본 차원 |
|---|---|
| `text-embedding-3-small` | 1536 |
| `text-embedding-3-large` | 3072 |

- 결과는 실수 리스트 하나라 `len(vector)`가 차원 수
- `dimensions`로 차원을 줄일 수 있음 (`text-embedding-3` 계열). 저장 공간과 계산은 줄지만 담기는 정보도 줄어듦

### 1-1. keras Embedding과 비교

| | keras `Embedding` | `OpenAIEmbeddings` |
|---|---|---|
| 벡터 길이 | `output_dim` | `dimensions` (기본 1536·3072) |
| 무엇을 벡터로 | 단어 번호 하나마다 | 문장 전체를 하나로 |
| 값을 정하는 방법 | 내 모델에서 처음부터 학습 | 학습된 모델이 만들어 줌 |
| 출력 | `(문장, 단어, output_dim)` | 길이 `dimensions`짜리 리스트 하나 |

- 둘 다 텍스트를 의미가 담긴 실수 벡터로 바꾼다는 점은 같음 (day22 §1)

### 1-2. 코사인 유사도

두 벡터 사이 각도로 유사한 정도를 재는 방식

![고양이 (0.8, 0.6), 강아지 (0.7, 0.7), 자동차 (-0.2, 0.9) 벡터를 2차원 평면에 그렸다. 고양이와 강아지 사이는 8도로 코사인 유사도 0.99, 고양이와 자동차 사이는 66도로 0.41이다](assets/cosine-similarity.svg)

임베딩 벡터는 뜻이 비슷할수록 비슷한 방향을 가리키므로, 각도를 보면 두 문장이 얼마나 비슷한지 숫자로 나옴

```
cos = (A · B) / (|A| × |B|)
```

1. 내적 `A · B` : 같은 자리끼리 곱해서 더함
2. 길이 `|A|` : 각 값을 제곱해서 더한 뒤 루트
3. 나누기 : 내적을 두 길이로 나눠 길이의 영향을 지움

| 각도 | cos | 뜻 |
|---|---|---|
| 0° | 1 | 방향이 같음 |
| 90° | 0 | 관계없음 |
| 180° | -1 | 정반대 |

- 각도 차가 클수록 유사도가 떨어짐
- 길이는 보지 않음. 방향이 같으면 길이가 달라도 1

2차원 예시

| A | B | 내적 | 길이 곱 | cos | 각도 |
|---|---|---|---|---|---|
| (1, 0) | (1, 1) | 1×1 + 0×1 = 1 | 1 × √2 ≈ 1.414 | 0.707 | 45° |
| (1, 0) | (0, 1) | 1×0 + 0×1 = 0 | 1 × 1 = 1 | 0 | 90° |
| (1, 0) | (-1, 0) | 1×(-1) + 0×0 = -1 | 1 × 1 = 1 | -1 | 180° |
| (1, 1) | (3, 3) | 1×3 + 1×3 = 6 | √2 × √18 = 6 | 1 | 0° |

- 마지막 줄 : `(3, 3)`은 `(1, 1)`보다 3배 길지만 방향이 같아서 1

단어 벡터로 보면

| 단어 | 벡터 |
|---|---|
| 고양이 | (0.8, 0.6) |
| 강아지 | (0.7, 0.7) |
| 자동차 | (-0.2, 0.9) |

1. 고양이 · 강아지 : `(0.8×0.7 + 0.6×0.7) / (1.0 × 0.990) = 0.98 / 0.990 ≈ 0.99`
2. 고양이 · 자동차 : `(0.8×(-0.2) + 0.6×0.9) / (1.0 × 0.922) = 0.38 / 0.922 ≈ 0.41`

- 실제 임베딩은 1536차원이지만 계산 방법은 같음. 자리 수만 많아짐